In [151]:
import biom
import pandas as pd
import numpy as np
from pathlib import Path

In [149]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

In [27]:
metadata = pd.read_csv("../../data/metadata_disease_classification.tsv", sep="\t", index_col="sample")

### weighted SNE

In [62]:
clf = LogisticRegression(penalty="l2", C=0.01,      
    max_iter=1000, random_state=42, n_jobs=1)

In [41]:
snes_embed = pd.read_csv("../../data/social_niche_embedding_removing_disease_samples_100.txt", sep=" ",index_col=0, header=None)
snes_embed.head(n=3)

,1,2,3,4,5,6,7,8,9,10,...,91,92,93,94,95,96,97,98,99,100
0,,,,,,,,,,,,,,,,,,,,,
AAAA02020714.1.1202,-0.418816,-0.000134,0.022425,-0.257710,1.048412,1.034152,-0.257349,-0.671971,0.161841,-0.888736,...,0.271436,-0.332842,0.177369,-0.125318,-0.738048,1.023876,-0.216218,0.799061,0.507492,0.288004
AAFJ01000001.39328.40836,-0.608513,-0.541320,0.537584,-0.032370,-0.441706,-0.295041,-0.101782,-0.289670,-0.346781,-0.168612,...,-0.559139,-0.051566,0.146513,-0.182651,0.444305,0.163505,-0.394681,0.618323,0.417238,-0.768848
AAQK01001555.694.2198,-1.931472,-0.352034,-0.656727,-0.015706,-0.513147,0.380205,-0.093266,0.650531,-0.320520,-0.737828,...,0.687433,-0.372768,0.422629,0.144155,0.172128,-0.557978,0.238239,-0.440044,0.329425,0.213950


### RF results

In [152]:
out_dir = Path("Data/disease_data/results_with_rf_embed")
out_dir.mkdir(parents=True, exist_ok=True)

#### CRC

In [138]:
model = RandomForestClassifier(random_state=11, n_jobs=1, n_estimators=200)

In [162]:
for i in ["PRJDB11845", "PRJEB36789", "PRJEB6070", "PRJNA290926", "PRJNA318004", "PRJNA430990", "PRJNA824020"]:
    out_dir = Path(f"Data/disease_data/results_with_rf_embed/CRC_{i}")
    out_dir.mkdir(parents=True, exist_ok=True)
    tr = biom.load_table(f"Data/disease_data/CRC/{i}/train_loo.biom")
    te = biom.load_table(f"Data/disease_data/CRC/{i}/test_loo.biom")
    fid = np.intersect1d(tr.ids(axis="observation"), te.ids(axis="observation"))
    fid = np.intersect1d(fid, snes_embed.index.values)
    tr.filter(fid, axis="observation", inplace=True)
    te.filter(fid, axis="observation", inplace=True)

    ytr = (metadata.loc[tr.ids(axis="sample"), 'group'] == 1).astype(int).values
    yte = (metadata.loc[te.ids(axis="sample"), 'group'] == 1).astype(int).values
    Xtr = tr.norm(axis="sample", inplace=False).matrix_data.toarray().T
    Xte = te.norm(axis="sample", inplace=False).matrix_data.toarray().T

    Xtr = Xtr @ snes_embed.loc[fid]
    Xte = Xte @ snes_embed.loc[fid]
   
    model.fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    proba = model.predict_proba(Xte)[:, 1]
    df = pd.DataFrame({
        "sample_id": te.ids(axis="sample"),
        "true_label": yte,
        "prob": proba,
    })
    df.to_csv(f"{out_dir}/pred_prob.csv", index=False)
    print(f"study: {i}; auc: {auc}")

study: PRJDB11845; auc: 0.67
study: PRJEB36789; auc: 0.6403306420607459
study: PRJEB6070; auc: 0.6768292682926829
study: PRJNA290926; auc: 0.5448615384615385
study: PRJNA318004; auc: 0.6404631083202512
study: PRJNA430990; auc: 0.5981948334889512
study: PRJNA824020; auc: 0.7196969696969697


#### IBD

In [161]:
for i in ["PRJNA324147", "PRJNA368966", "PRJNA422193", "PRJNA431126", "PRJNA450340", "RISK_PRISM_f", "qiita_1629", "qiita_2538"]:

    out_dir = Path(f"Data/disease_data/results_with_rf_embed/IBD_{i}")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    tr = biom.load_table(f"Data/disease_data/IBD/{i}/train_loo.biom")
    te = biom.load_table(f"Data/disease_data/IBD/{i}/test_loo.biom")
    fid = np.intersect1d(tr.ids(axis="observation"), te.ids(axis="observation"))
    fid = np.intersect1d(fid, snes_embed.index.values)
    tr.filter(fid, axis="observation", inplace=True)
    te.filter(fid, axis="observation", inplace=True)

    ytr = (metadata.loc[tr.ids(axis="sample"), 'group'] == 1).astype(int).values
    yte = (metadata.loc[te.ids(axis="sample"), 'group'] == 1).astype(int).values
    Xtr = tr.norm(axis="sample", inplace=False).matrix_data.toarray().T
    Xte = te.norm(axis="sample", inplace=False).matrix_data.toarray().T

    Xtr = Xtr @ snes_embed.loc[fid]
    Xte = Xte @ snes_embed.loc[fid]

    model.fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    proba = model.predict_proba(Xte)[:, 1]
    df = pd.DataFrame({
        "sample_id": te.ids(axis="sample"),
        "true_label": yte,
        "prob": proba,
    })
    df.to_csv(f"{out_dir}/pred_prob.csv", index=False)
    print(f"study: {i}; auc: {auc}")

study: PRJNA324147; auc: 0.6520215633423181
study: PRJNA368966; auc: 0.6412512218963833
study: PRJNA422193; auc: 0.7483883964544722
study: PRJNA431126; auc: 0.5724342928660827
study: PRJNA450340; auc: 0.6647879763821792
study: RISK_PRISM_f; auc: 0.5098283499446291
study: qiita_1629; auc: 0.5796231937077008
study: qiita_2538; auc: 0.5824617956064948
